# Volume 1 Drills — Mathematical Foundations

15 short exercises covering the mathematics of Markov Decision Processes,  
Bellman equations, policy evaluation, and dynamic programming.

**Prerequisites:** Python basics, Volume 1 reading (or bridge exercises).

---
## Drill 1 — Discounted Return by Hand, Then Code

**By hand first:** For rewards `[2, 0, -1]` with γ = 0.9, compute G₀.

$$G_0 = 2 + 0.9 \times 0 + 0.9^2 \times (-1) = ?$$

Now verify with code:

In [ ]:
def discounted_return(rewards, gamma=0.9):
    G = 0.0
    for r in reversed(rewards):
        G = r + gamma * G
    return G

# Verify your hand calculation:
print(discounted_return([2, 0, -1], gamma=0.9))
# Expected: 2 + 0 + 0.81*(-1) = 1.19

# TODO: What is G_0 for rewards=[1, 1, 1, 1, 1] with gamma=0.5?
# Compute by hand: ________________
# Verify:
print(discounted_return([1, 1, 1, 1, 1], gamma=0.5))

---
## Drill 2 — Bellman Equation for Two States

The Bellman equation for policy π:

$$V^\pi(s) = \sum_{s'} P(s'|s) \left[ R(s,s') + \gamma V^\pi(s') \right]$$

Given a 2-state MDP:
- From **s0**: go to s0 with prob 0.7 (reward 0), go to s1 with prob 0.3 (reward 0)
- From **s1**: go to s0 with prob 0.4 (reward 0), go to s1 with prob 0.6 (reward 0)
- Terminal reward: V(s0) and V(s1) are seeded with [1.0, 0.5]

Compute one Bellman backup step:

In [ ]:
gamma = 0.9
V = [1.0, 0.5]   # current estimates

# Transition: P[s][s'] = probability, R[s][s'] = reward
P = [[0.7, 0.3],   # from s0
     [0.4, 0.6]]   # from s1
R = [[0.0, 0.0],
     [0.0, 0.0]]

def bellman_backup(V, P, R, gamma):
    """One step of Bellman backup for all states."""
    n = len(V)
    V_new = [0.0] * n
    for s in range(n):
        V_new[s] = sum(P[s][sp] * (R[s][sp] + gamma * V[sp])
                       for sp in range(n))
    return V_new

V_new = bellman_backup(V, P, R, gamma)
print(f"V(s0): {V_new[0]:.4f}")
print(f"V(s1): {V_new[1]:.4f}")

# TODO: Run 20 iterations — what do the values converge to?
V_iter = V[:]
for _ in range(20):
    V_iter = bellman_backup(V_iter, P, R, gamma)
print(f"After 20 iterations: {[f'{v:.4f}' for v in V_iter]}")

---
## Drill 3 — Iterative Policy Evaluation (One Sweep)

Implement **one sweep** of iterative policy evaluation on a 3-state MDP.  
A "sweep" updates every state once using the current V estimates.

MDP: states {0, 1, 2}, state 2 is terminal (V=0 always).  
Policy π: always move to the next state (0→1, 1→2).  
Rewards: r(0→1)=1, r(1→2)=1.

In [ ]:
def policy_evaluation_sweep(V, gamma=0.9):
    """One sweep of policy evaluation.
    States: 0, 1, 2 (terminal).
    Policy: state 0 -> state 1 (r=1), state 1 -> state 2 (r=1).
    """
    V_new = V[:]
    # State 0: transitions deterministically to state 1 with reward 1
    V_new[0] = 1 + gamma * V[1]
    # State 1: transitions deterministically to state 2 with reward 1
    V_new[1] = 1 + gamma * V[2]
    # State 2: terminal, V stays 0
    V_new[2] = 0.0
    return V_new

V = [0.0, 0.0, 0.0]
print(f"Initial V:  {V}")
for sweep in range(1, 6):
    V = policy_evaluation_sweep(V)
    print(f"Sweep {sweep}:   {[f'{v:.4f}' for v in V]}")

# True values: V(0) = 1 + gamma*1 = 1+0.9 = 1.9, V(1) = 1, V(2) = 0
print("\nExpected converged: V(0)≈1.9, V(1)≈1.0, V(2)=0.0")

---
## Drill 4 — Value Iteration (One Step)

Value iteration uses the **Bellman optimality operator**:

$$V_{k+1}(s) = \max_a \sum_{s'} P(s'|s,a) \left[ R(s,a,s') + \gamma V_k(s') \right]$$

Implement one step for a small MDP where each state has 2 actions.

In [ ]:
# 2-state MDP, 2 actions each
# P[s][a][s'] = transition probability
# R[s][a] = immediate reward
P = {
    0: {0: {0: 0.8, 1: 0.2}, 1: {0: 0.1, 1: 0.9}},
    1: {0: {0: 0.7, 1: 0.3}, 1: {0: 0.4, 1: 0.6}},
}
R = {0: {0: 5, 1: 10}, 1: {0: -1, 1: 2}}
gamma = 0.9

def value_iteration_step(V, P, R, gamma):
    """One step of value iteration. Returns updated V and greedy policy."""
    V_new = {}
    policy = {}
    for s in V:
        action_values = {}
        for a in P[s]:
            action_values[a] = R[s][a] + gamma * sum(
                P[s][a][sp] * V[sp] for sp in P[s][a]
            )
        best_a = max(action_values, key=action_values.get)
        V_new[s] = action_values[best_a]
        policy[s] = best_a
    return V_new, policy

V = {0: 0.0, 1: 0.0}
for k in range(1, 11):
    V, pi = value_iteration_step(V, P, R, gamma)
    print(f"Step {k:2d}: V={V}  π={pi}")

---
## Drill 5 — Debug the Broken `discounted_return`

The function below has a bug. Find it and fix it.

**Hint:** Check the output — does it match the expected value?

In [ ]:
# BUGGY version — find and fix the bug
def discounted_return_buggy(rewards, gamma=0.9):
    G = 0.0
    for t, r in enumerate(rewards):
        G += r   # BUG: forgot to multiply by gamma^t
    return G

# This should return 0.729 (0.9^3 * 1), but what does the buggy version return?
print("Buggy output:", discounted_return_buggy([0, 0, 0, 1]))

# TODO: Write the fixed version below:
def discounted_return_fixed(rewards, gamma=0.9):
    # TODO: fix the bug
    pass

print("Fixed output:", discounted_return_fixed([0, 0, 0, 1]))  # should be 0.729

<details><summary>▶ Solution</summary>

```python
def discounted_return_fixed(rewards, gamma=0.9):
    G = 0.0
    for t, r in enumerate(rewards):
        G += (gamma ** t) * r  # FIX: include gamma^t
    return G
# Or equivalently, work backwards:
def discounted_return_fixed(rewards, gamma=0.9):
    G = 0.0
    for r in reversed(rewards):
        G = r + gamma * G
    return G
```
</details>

---
## Drill 6 — Bellman Backup for a Gridworld Cell

Consider a 3×3 gridworld. The current value estimates are:

```
V = [[0.0,  1.0,  0.0],
     [0.5,  ???,  0.5],
     [0.0,  1.0,  0.0]]
```

For state **(1,1)** (center), the **random policy** moves equally to all 4 neighbors.  
Reward is 0 for all moves except reaching (0,1) or (2,1) which give reward 1.  
Compute V(1,1) after one Bellman backup with γ = 0.9.

In [ ]:
import numpy as np

V = np.array([[0.0, 1.0, 0.0],
              [0.5, 0.0, 0.5],
              [0.0, 1.0, 0.0]])
gamma = 0.9

# Neighbors of (1,1): up=(0,1), down=(2,1), left=(1,0), right=(1,2)
neighbors = [(0,1), (2,1), (1,0), (1,2)]
# Rewards for moving to each neighbor (1 if the neighbor is (0,1) or (2,1))
rewards = [1.0, 1.0, 0.0, 0.0]

# TODO: compute V(1,1) = (1/4) * sum over neighbors of (r + gamma * V(neighbor))
V_center = sum(1/4 * (rewards[i] + gamma * V[neighbors[i]])
               for i in range(4))

print(f"Bellman backup V(1,1) = {V_center:.4f}")
# Expected: 0.25 * [(1+0.9*1) + (1+0.9*1) + (0+0.9*0.5) + (0+0.9*0.5)]
#         = 0.25 * [1.9 + 1.9 + 0.45 + 0.45] = 0.25 * 4.7 = 1.175
print("Expected: 1.175")

---
## Drill 7 — Debug: Wrong ε-Greedy Direction

The code below has a subtle but critical bug in the exploration condition. Find it.

In [ ]:
import random

# BUGGY epsilon-greedy
def epsilon_greedy_buggy(q_values, epsilon=0.1):
    """q_values: list of Q(s,a) for each action."""
    if random.random() > epsilon:   # BUG: should be < epsilon for random exploration
        return random.randint(0, len(q_values) - 1)  # random
    else:
        return q_values.index(max(q_values))          # greedy

# Test: with epsilon=0.1, greedy action (index 2, value 5.0) should dominate
random.seed(42)
q = [1.0, 2.0, 5.0, 0.5]
counts = [0] * 4
for _ in range(1000):
    a = epsilon_greedy_buggy(q, epsilon=0.1)
    counts[a] += 1
print("Buggy counts:", counts)
print("Expected: action 2 should appear ~900/1000 times, not ~100/1000")

# TODO: Write the fixed version
def epsilon_greedy_fixed(q_values, epsilon=0.1):
    # TODO: fix the comparison direction
    pass

random.seed(42)
counts_fixed = [0] * 4
for _ in range(1000):
    a = epsilon_greedy_fixed(q, epsilon=0.1)
    if a is not None:
        counts_fixed[a] += 1
print("Fixed counts:", counts_fixed)

<details><summary>▶ Solution</summary>

The bug: `random.random() > epsilon` means the random branch executes **90%** of the time (when random > 0.1), and the greedy branch only **10%**. It should be `< epsilon`:

```python
def epsilon_greedy_fixed(q_values, epsilon=0.1):
    if random.random() < epsilon:  # FIX: < not >
        return random.randint(0, len(q_values) - 1)
    else:
        return q_values.index(max(q_values))
```
</details>

---
## Drill 8 — MDP Formulation

**Conceptual:** For each scenario below, identify the **(S, A, P, R, γ)** components.

In [ ]:
# Scenario: A robot navigates a 4x4 grid. It wants to reach a goal.
# Each step costs -0.04 (small negative reward). Reaching goal = +1. Falling off = -1.

# Fill in:
mdp_components = {
    "S": "TODO: describe state space",       # e.g., (row, col) pairs
    "A": "TODO: describe action space",      # e.g., {up, down, left, right}
    "P": "TODO: describe transition model",  # deterministic? stochastic?
    "R": "TODO: describe reward function",   # step cost + terminal rewards
    "gamma": "TODO: choose and justify γ",   # e.g., 0.99 for long-horizon tasks
}

for k, v in mdp_components.items():
    print(f"{k:6s}: {v}")

---
## Drill 9 — Policy Improvement

Given V(s) from policy evaluation, extract the **greedy policy** (policy improvement step).

In [ ]:
# 3-state MDP: states 0,1,2. Action 0 = stay, action 1 = move forward.
# Transition: action 1 always moves to (state+1) % 3.
# Rewards: R(s=1, a=1) = 10 (reaching state 2), else 0.
V = {0: 5.0, 1: 8.0, 2: 0.0}   # from some prior evaluation
gamma = 0.9

def policy_improvement(V, gamma=0.9):
    """For each state, find the action that maximizes R + gamma*V(s')."""
    policy = {}
    for s in [0, 1, 2]:
        # action 0: stay in s, reward 0
        q_stay = 0 + gamma * V[s]
        # action 1: move to (s+1)%3, reward 10 if s==1 else 0
        s_next = (s + 1) % 3
        reward = 10 if s == 1 else 0
        q_move = reward + gamma * V[s_next]
        policy[s] = 0 if q_stay >= q_move else 1
    return policy

pi = policy_improvement(V)
print("Greedy policy:", pi)
print("Interpretation: 0=stay, 1=move")

---
## Drill 10 — Convergence Check

Value iteration converges when **max|V_new(s) − V_old(s)|** < threshold.  
Implement a full value iteration loop with convergence detection.

In [ ]:
def value_iteration(P, R, gamma=0.9, theta=1e-6, max_iter=1000):
    """Full value iteration with convergence check.
    P[s][a][s'] = prob, R[s][a] = reward, returns (V, policy, n_iters).
    """
    states = list(P.keys())
    V = {s: 0.0 for s in states}
    for k in range(max_iter):
        delta = 0.0
        for s in states:
            v_old = V[s]
            V[s] = max(
                R[s][a] + gamma * sum(P[s][a][sp] * V[sp] for sp in P[s][a])
                for a in P[s]
            )
            delta = max(delta, abs(V[s] - v_old))
        if delta < theta:
            return V, k + 1
    return V, max_iter

# Reuse MDP from Drill 4
P = {
    0: {0: {0: 0.8, 1: 0.2}, 1: {0: 0.1, 1: 0.9}},
    1: {0: {0: 0.7, 1: 0.3}, 1: {0: 0.4, 1: 0.6}},
}
R = {0: {0: 5, 1: 10}, 1: {0: -1, 1: 2}}

V_star, n_iters = value_iteration(P, R)
print(f"Converged in {n_iters} iterations")
print(f"V* = {V_star}")

---
## Drill 11 — Q-Value from V-Value

Given V*(s), compute Q*(s,a) using:

$$Q^*(s,a) = R(s,a) + \gamma \sum_{s'} P(s'|s,a) V^*(s')$$

In [ ]:
def compute_q_from_v(V, P, R, gamma=0.9):
    """Compute Q(s,a) from V, P, R."""
    Q = {}
    for s in P:
        for a in P[s]:
            Q[(s, a)] = R[s][a] + gamma * sum(
                P[s][a][sp] * V.get(sp, 0.0) for sp in P[s][a]
            )
    return Q

Q = compute_q_from_v(V_star, P, R)
for (s, a), q in sorted(Q.items()):
    print(f"Q*({s},{a}) = {q:.4f}")

---
## Drill 12 — Stochastic Policy

A stochastic policy π(a|s) assigns probabilities to actions.  
The Bellman equation becomes:

$$V^\pi(s) = \sum_a \pi(a|s) \sum_{s'} P(s'|s,a) [R(s,a,s') + \gamma V^\pi(s')]$$

Implement one evaluation sweep for a uniform random policy.

In [ ]:
def policy_eval_stochastic(V, pi, P, R, gamma=0.9):
    """One Bellman sweep under stochastic policy pi.
    pi[s][a] = probability of action a in state s.
    """
    V_new = dict(V)
    for s in P:
        V_new[s] = sum(
            pi[s][a] * (
                R[s][a] + gamma * sum(P[s][a][sp] * V[sp] for sp in P[s][a])
            )
            for a in P[s]
        )
    return V_new

# Uniform random policy over 2 actions
pi_uniform = {s: {a: 0.5 for a in P[s]} for s in P}
V = {0: 0.0, 1: 0.0}
for k in range(50):
    V = policy_eval_stochastic(V, pi_uniform, P, R)
print(f"V under uniform policy: {V}")

---
## Drill 13 — Why Discount?

**Conceptual + code:** Explore what happens to the return as γ changes.

In [ ]:
# Infinite geometric series: if rewards = [1, 1, 1, ...] forever,
# G = sum(gamma^t) = 1 / (1 - gamma)  (for gamma < 1)

def approx_infinite_return(gamma, n_steps=1000):
    """Approximate infinite return with constant reward=1."""
    return sum(gamma**t for t in range(n_steps))

for g in [0.5, 0.9, 0.99, 0.999]:
    approx = approx_infinite_return(g)
    exact = 1 / (1 - g)
    print(f"gamma={g:.3f}: approx={approx:.4f}  exact={exact:.4f}")

print("\nWith gamma=1.0, the sum diverges — that's why we need gamma < 1!")

---
## Drill 14 — Policy Iteration Loop

Combine policy evaluation + policy improvement into one **policy iteration** loop.

In [ ]:
def policy_iteration(P, R, gamma=0.9, eval_iters=100):
    """Full policy iteration: alternates evaluation and improvement."""
    states = list(P.keys())
    actions = {s: list(P[s].keys()) for s in states}

    # Start with arbitrary policy (always action 0)
    pi = {s: 0 for s in states}
    V = {s: 0.0 for s in states}

    for iteration in range(20):
        # --- Policy Evaluation ---
        for _ in range(eval_iters):
            for s in states:
                a = pi[s]
                V[s] = R[s][a] + gamma * sum(P[s][a][sp] * V[sp] for sp in P[s][a])

        # --- Policy Improvement ---
        policy_stable = True
        for s in states:
            old_a = pi[s]
            pi[s] = max(actions[s],
                        key=lambda a: R[s][a] + gamma * sum(P[s][a][sp] * V[sp] for sp in P[s][a]))
            if old_a != pi[s]:
                policy_stable = False

        if policy_stable:
            print(f"Policy stable after {iteration + 1} iterations")
            break

    return V, pi

V_pi, pi_star = policy_iteration(P, R)
print(f"V = {V_pi}")
print(f"π* = {pi_star}")
print(f"V* = {V_star}  (should match)")

---
## Drill 15 — The Bellman Optimality Equation

**Fill in the blanks:** Explain each term in the Bellman optimality equation.

$$V^*(s) = \max_a \left[ \underbrace{R(s,a)}_{?} + \gamma \sum_{s'} \underbrace{P(s'|s,a)}_{?} \underbrace{V^*(s')}_{?} \right]$$

In [ ]:
# Fill in your answers as strings:
answers = {
    "R(s,a)": "TODO: what does this term represent?",
    "P(s'|s,a)": "TODO: what does this term represent?",
    "V*(s')": "TODO: what does this term represent?",
    "max_a": "TODO: why do we take the max over actions?",
    "gamma": "TODO: what role does the discount factor play?",
}

print("Bellman Optimality — Your Answers:")
for term, answer in answers.items():
    print(f"  {term:12s}: {answer}")

<details><summary>▶ Reference Answers</summary>

- **R(s,a)**: Immediate reward received when taking action `a` in state `s`.
- **P(s'|s,a)**: Transition probability — how likely we land in state `s'` after taking action `a` in state `s`. Captures environment stochasticity.
- **V*(s')**: Optimal future return from the next state — this is the "bootstrap" target.
- **max_a**: We're finding the *optimal* policy, so we choose the action that maximizes total expected return.
- **gamma**: Discounts future rewards. Makes the sum finite for infinite-horizon problems and encodes preference for sooner rewards.
</details>

---
## Summary

| Drill | Topic |
|-------|-------|
| 1 | Discounted return G_t |
| 2 | Bellman backup (tabular) |
| 3 | Iterative policy evaluation |
| 4 | Value iteration step |
| 5 | Debugging return function |
| 6 | Gridworld Bellman backup |
| 7 | Debugging ε-greedy |
| 8 | MDP components formulation |
| 9 | Policy improvement (greedy) |
| 10 | Full value iteration |
| 11 | Q from V |
| 12 | Stochastic policy evaluation |
| 13 | Why γ < 1 matters |
| 14 | Policy iteration loop |
| 15 | Bellman optimality equation |

**Next:** Volume 2 — Tabular Methods (TD learning, Q-learning, SARSA).